# Programación orientada a objetos en Python

En este notebook aprenderás a crear tus propias clases en Python. Si vienes de Java, verás que las ideas son las mismas (clases, objetos, herencia, polimorfismo), pero con mucha menos ceremonia: sin `public` ni `private`, sin *getters* y *setters* escritos a mano y sin declarar los atributos. Al final usarás *dataclasses*, `Enum` y anotaciones de tipo, que son la forma moderna de escribir clases en Python.

**Cómo usarlo:**
- Ejecuta cada celda con `Shift+Intro`, **en orden**.
- En los **ejercicios**, sustituye los `...` por tu código.
- Ejecuta la celda de **comprobación** de debajo: si aparece `Correcto`, lo tienes. Si aparece un `AssertionError`, revisa tu solución.
- Muchas celdas **vuelven a definir** una clase para ampliarla. Si una celda falla con un `NameError` o se comporta de forma rara, ejecuta de nuevo las celdas anteriores.

## 0. Antes de empezar

Comprueba la versión de Python. Necesitas Python 3.11 o posterior.

In [ ]:
import sys

print(sys.version)
assert sys.version_info >= (3, 11), "Necesitas Python 3.11 o posterior"

## 1. De Java a Python

Una **clase** es una plantilla y un **objeto** es un ejemplar creado a partir de ella. Hasta aquí, nada nuevo. Lo que cambia es la sintaxis:

| Java | Python |
|------|--------|
| `public class Bicicleta { ... }` | `class Bicicleta:` |
| Constructor `Bicicleta(String id)` | Método `__init__(self, id)` |
| `this.id = id;` | `self.id = id` |
| Los atributos se declaran arriba, con su tipo | Los atributos se crean al asignarlos, normalmente en `__init__` |
| `new Bicicleta("B-01")` | `Bicicleta("B-01")`, sin `new` |
| `public`, `private`, `protected` | No existen. Se usan convenciones con `_` |
| *Getters* y *setters* | Acceso directo al atributo y, si hace falta, `@property` |

In [ ]:
class Bicicleta:
    def __init__(self, id, km=0):
        self.id = id          # atributo de instancia
        self.km = km

    def recorrer(self, km):
        self.km += km
        return self.km


b1 = Bicicleta("B-01")
b2 = Bicicleta("B-02", km=120)

b1.recorrer(5)
b1.recorrer(3.5)
print(b1.id, b1.km)
print(b2.id, b2.km)

**`self` es explícito.** En Java, `this` está siempre disponible sin declararlo. En Python, el objeto sobre el que se llama un método llega como **primer parámetro**, y por convención se llama `self`. Por eso estas dos líneas son equivalentes:

In [ ]:
b1.recorrer(10)               # la forma habitual
Bicicleta.recorrer(b1, 10)    # lo que hace Python por debajo: b1 llega como self
print(b1.km)

**Sin modificadores de acceso.** En Python todo es público. Para indicar que algo es "interno" se usan convenciones:

- `_nombre` (un guion bajo): "esto es interno, no lo uses desde fuera". Nadie te lo impide, pero es un aviso para quien lea el código.
- `__nombre` (dos guiones bajos): Python le cambia el nombre por debajo (*name mangling*) a `_Clase__nombre`, para evitar choques en las subclases. No es privacidad real.

> "Somos todos adultos": Python confía en que respetes las convenciones.

In [ ]:
class Candado:
    def __init__(self, codigo):
        self._intentos = 0        # interno, por convención
        self.__codigo = codigo    # Python lo renombra a _Candado__codigo

    def abrir(self, codigo):
        self._intentos += 1
        return codigo == self.__codigo


c = Candado(1234)
print(c.abrir(1111), c.abrir(1234), c._intentos)

#print(c.__codigo)            # descomenta: AttributeError
print(c._Candado__codigo)      # ...pero sigue ahí: no es privado de verdad
print(vars(c))                 # todos los atributos del objeto, en un diccionario

**En Python todo es un objeto.** Los números, las cadenas, las listas, las funciones e incluso las propias clases son objetos: tienen un tipo, atributos y métodos.

In [ ]:
print(type(5), type("hola"), type([1, 2]), type(len), type(Bicicleta))

print((5).bit_length())        # los enteros tienen métodos
print((2.5).is_integer())
print("hola".upper())

def saludar(nombre):
    return f"Hola, {nombre}"

decir = saludar                # una función se puede guardar en una variable...
print(decir("Ana"))
print(saludar.__name__)        # ...y tiene atributos

print(isinstance(b1, Bicicleta), isinstance(5, int), isinstance(True, int))

### Ejercicio 1.1

Crea la clase `Usuario` de un servicio de bicicletas:
- `__init__(self, nombre, saldo=0)` guarda el nombre y el saldo.
- `recargar(self, cantidad)` suma la cantidad al saldo y **devuelve** el saldo nuevo.
- `pagar(self, cantidad)` resta la cantidad y devuelve el saldo nuevo. Si no hay saldo suficiente, lanza un `ValueError` **sin descontar nada**.

In [ ]:
class Usuario:
    def __init__(self, nombre, saldo=0):
        ...

    def recargar(self, cantidad):
        ...

    def pagar(self, cantidad):
        ...


ana = Usuario("Ana")
print(ana.recargar(10), ana.pagar(3.5))

In [ ]:
# Comprobación
u = Usuario("Luis")
assert u.nombre == "Luis" and u.saldo == 0
assert Usuario("Eva", saldo=5).saldo == 5
assert u.recargar(10) == 10, "recargar() debe devolver el saldo nuevo"
assert u.pagar(3.5) == 6.5, "pagar() debe devolver el saldo nuevo"
try:
    u.pagar(100)
    raise AssertionError("Pagar más de lo que hay debe lanzar ValueError")
except ValueError:
    pass
assert u.saldo == 6.5, "Si no hay saldo suficiente, no se debe descontar nada"
print("Correcto")

## 2. Clases

### Atributos de instancia y de clase

- Un **atributo de instancia** (`self.km`) es de cada objeto: cada bicicleta tiene sus kilómetros.
- Un **atributo de clase** se define dentro de la clase, fuera de los métodos, y lo **comparten** todos los objetos. Equivale a un `static` de Java. Se usa para constantes y contadores.

In [ ]:
class Bicicleta:
    TARIFA_MINUTO = 0.05      # atributo de clase: igual para todas
    creadas = 0               # contador compartido

    def __init__(self, id, km=0):
        self.id = id
        self.km = km
        Bicicleta.creadas += 1

    def tarifa(self, minutos):
        return round(minutos * self.TARIFA_MINUTO, 2)


a = Bicicleta("B-01")
b = Bicicleta("B-02")
print(Bicicleta.creadas, a.TARIFA_MINUTO, a.tarifa(20))

**Cuidado con los atributos de clase modificables.** Si un atributo de clase es una lista o un diccionario, **todos los objetos comparten la misma**. Las colecciones de cada objeto se crean siempre en `__init__`.

In [ ]:
class EstacionMal:
    bicis = []                        # una sola lista para TODAS las estaciones

    def __init__(self, nombre):
        self.nombre = nombre


class EstacionBien:
    def __init__(self, nombre):
        self.nombre = nombre
        self.bicis = []               # una lista nueva para cada estación


m1, m2 = EstacionMal("Alameda"), EstacionMal("Soho")
m1.bicis.append("B-01")
print("Mal: ", m2.bicis)              # ¡Soho también tiene la B-01!

b1, b2 = EstacionBien("Alameda"), EstacionBien("Soho")
b1.bicis.append("B-01")
print("Bien:", b2.bicis)

### Propiedades: `@property`

En Java escribirías `getCapacidad()` y `setCapacidad()` "por si acaso". En Python se accede **directamente** al atributo y, si más adelante hace falta calcular o validar algo, se convierte en una **propiedad** sin cambiar el código que la usa.

- Con `@property`, un método se usa **como si fuera un atributo**: `estacion.libres`, sin paréntesis.
- Con `@nombre.setter`, se valida el valor al asignarlo.

In [ ]:
class Estacion:
    def __init__(self, nombre, capacidad, bicis=0):
        self.nombre = nombre
        self.capacidad = capacidad
        self.bicis = bicis            # pasa por el setter de abajo

    @property
    def libres(self):                 # propiedad calculada, de solo lectura
        return self.capacidad - self.bicis

    @property
    def bicis(self):
        return self._bicis

    @bicis.setter
    def bicis(self, valor):
        if not 0 <= valor <= self.capacidad:
            raise ValueError(f"{self.nombre}: {valor} bicis no caben en {self.capacidad} anclajes")
        self._bicis = valor


e = Estacion("Alameda Principal", capacidad=30, bicis=12)
print(e.bicis, e.libres)

e.bicis = 25                          # parece una asignación normal, pero valida
print(e.libres)

try:
    e.bicis = 40
except ValueError as error:
    print("Error:", error)

### Métodos de clase y métodos estáticos

- `@classmethod` recibe la **clase** (`cls`) en lugar del objeto. Se usa sobre todo para **constructores alternativos**, como crear un objeto a partir de un diccionario. Python no tiene sobrecarga de constructores como Java: esta es la forma de tener varios.
- `@staticmethod` no recibe ni el objeto ni la clase. Es una función normal que vive dentro de la clase porque tiene que ver con ella.

In [ ]:
class Estacion:
    def __init__(self, nombre, capacidad):
        self.nombre = nombre
        self.capacidad = capacidad

    @classmethod
    def desde_dict(cls, datos):
        return cls(datos["nombre"], int(datos["capacidad"]))

    @staticmethod
    def es_nombre_valido(nombre):
        return bool(nombre.strip())


fila = {"nombre": "Muelle Uno", "capacidad": "20"}      # como viene de un CSV
e = Estacion.desde_dict(fila)
print(e.nombre, e.capacidad, type(e.capacidad))
print(Estacion.es_nombre_valido("   "))

### Ejercicio 2.1

Completa la clase `Temperatura`, que guarda los grados en Celsius:
- En el *setter* de `celsius`, lanza un `ValueError` si el valor es menor que el cero absoluto (`-273.15`).
- Añade la propiedad `fahrenheit` (de solo lectura): `celsius * 9 / 5 + 32`.
- Añade el método de clase `desde_fahrenheit(cls, grados)`, que crea una `Temperatura` a partir de grados Fahrenheit: `(grados - 32) * 5 / 9`.

In [ ]:
class Temperatura:
    def __init__(self, celsius):
        self.celsius = celsius

    @property
    def celsius(self):
        return self._celsius

    @celsius.setter
    def celsius(self, valor):
        ...
        self._celsius = valor

    # Añade aquí la propiedad fahrenheit y el método de clase desde_fahrenheit


t = Temperatura(25)
print(t.celsius, t.fahrenheit)

In [ ]:
# Comprobación
assert Temperatura(100).fahrenheit == 212
assert Temperatura(-40).fahrenheit == -40
assert isinstance(Temperatura.__dict__.get("fahrenheit"), property), "fahrenheit debe ser una @property"
assert round(Temperatura.desde_fahrenheit(212).celsius, 6) == 100
assert isinstance(Temperatura.desde_fahrenheit(50), Temperatura)
for crear in (lambda: Temperatura(-300), lambda: setattr(Temperatura(0), "celsius", -274)):
    try:
        crear()
        raise AssertionError("Por debajo de -273.15 ºC debe lanzar ValueError")
    except ValueError:
        pass
print("Correcto")

### Métodos especiales (*dunder*)

Los métodos con doble guion bajo al principio y al final (*double underscore*, de ahí *dunder*) permiten que tus objetos funcionen con la sintaxis de Python: `print`, `==`, `<`, `len`, `for`, `in`... Son el equivalente a `toString()`, `equals()` o `compareTo()` de Java, pero mucho más amplios.

| Método | Se usa con | Equivale en Java a |
|--------|------------|--------------------|
| `__repr__` | `repr(x)`, la consola, el depurador | `toString()` (para programadores) |
| `__str__` | `print(x)`, `str(x)`, f-strings | `toString()` (para usuarios) |
| `__eq__` | `x == y` | `equals()` |
| `__lt__` | `x < y`, `sorted`, `min`, `max` | `compareTo()` |
| `__len__` | `len(x)` | `size()` |
| `__iter__` | `for e in x` | `Iterable` |
| `__contains__` | `e in x` | `contains()` |
| `__getitem__` | `x[i]` | `get(i)` |

In [ ]:
class Trayecto:
    def __init__(self, origen, destino, minutos):
        self.origen = origen
        self.destino = destino
        self.minutos = minutos

    def __repr__(self):
        return f"Trayecto({self.origen!r}, {self.destino!r}, {self.minutos})"

    def __str__(self):
        return f"{self.origen} → {self.destino} ({self.minutos} min)"

    def __eq__(self, otro):
        if not isinstance(otro, Trayecto):
            return NotImplemented
        return (self.origen, self.destino, self.minutos) == (otro.origen, otro.destino, otro.minutos)

    def __lt__(self, otro):
        return self.minutos < otro.minutos


t1 = Trayecto("Alameda", "Soho", 7)
t2 = Trayecto("Muelle Uno", "El Palo", 25)
t3 = Trayecto("Alameda", "Soho", 7)

print(t1)                         # usa __str__
print([t1, t2])                   # dentro de una lista se usa __repr__
print(t1 == t3, t1 is t3)         # iguales, pero no el mismo objeto
print(sorted([t2, t1]), max(t1, t2))

Y para que un objeto se comporte como una **colección**:

In [ ]:
class Ruta:
    def __init__(self, nombre, paradas):
        self.nombre = nombre
        self._paradas = list(paradas)

    def __len__(self):
        return len(self._paradas)

    def __iter__(self):
        return iter(self._paradas)

    def __contains__(self, parada):
        return parada in self._paradas

    def __getitem__(self, i):
        return self._paradas[i]


ruta = Ruta("Litoral", ["Huelin", "Alameda Principal", "Muelle Uno", "La Malagueta", "El Palo"])
print(len(ruta), "Muelle Uno" in ruta, ruta[0], ruta[-1], ruta[1:3])
for parada in ruta:
    print("-", parada)

### Ejercicio 2.2

Completa la clase `Dinero`, que guarda las cantidades en **céntimos** (un entero) para evitar los errores de redondeo de los `float`:
- `__repr__` devuelve `Dinero(12.50)`, siempre con dos decimales.
- `__str__` devuelve `12,50 €`, con coma decimal.
- `__add__` permite sumar dos `Dinero` con `+` y devuelve un `Dinero` nuevo.
- `__eq__` y `__lt__` comparan por céntimos.

In [ ]:
class Dinero:
    def __init__(self, euros):
        self.centimos = round(euros * 100)

    def __repr__(self):
        ...

    def __str__(self):
        ...

    def __add__(self, otro):
        ...

    def __eq__(self, otro):
        ...

    def __lt__(self, otro):
        ...


print(0.1 + 0.2)
print(Dinero(0.1) + Dinero(0.2))

In [ ]:
# Comprobación
assert repr(Dinero(12.5)) == "Dinero(12.50)"
assert str(Dinero(12.5)) == "12,50 €"
assert f"{Dinero(3)}" == "3,00 €"
suma = Dinero(0.1) + Dinero(0.2)
assert isinstance(suma, Dinero), "__add__ debe devolver un Dinero"
assert suma == Dinero(0.3), "Con céntimos, 0,10 + 0,20 sí da 0,30"
assert Dinero(1) != Dinero(2) and Dinero(1) < Dinero(2)
assert sorted([Dinero(5), Dinero(1.5), Dinero(3)]) == [Dinero(1.5), Dinero(3), Dinero(5)]
print("Correcto")

> **Nota:** al definir `__eq__`, Python deja de poder usar el objeto como clave de un diccionario o dentro de un conjunto (su `__hash__` pasa a ser `None`). Si lo necesitas, define también `__hash__`. Las *dataclasses* del apartado 4 lo resuelven por ti.

## 3. Herencia y diseño

### Herencia y `super()`

Se hereda poniendo la clase padre entre paréntesis: `class Hija(Padre)`. Con `super()` se llama a los métodos del padre, igual que en Java. En Python no existe `@Override`: basta con definir un método con el mismo nombre.

In [ ]:
class BicicletaElectrica(Bicicleta):
    DESBLOQUEO = 0.50

    def __init__(self, id, bateria=100, km=0):
        super().__init__(id, km)          # inicializa la parte de Bicicleta
        self.bateria = bateria

    def tarifa(self, minutos):            # sobrescribe el método del padre
        return round(super().tarifa(minutos) + self.DESBLOQUEO, 2)


normal = Bicicleta("B-10")
electrica = BicicletaElectrica("E-01", bateria=80)

print(normal.tarifa(20), electrica.tarifa(20))
print(electrica.id, electrica.bateria, electrica.TARIFA_MINUTO)     # hereda atributos
print(isinstance(electrica, Bicicleta), issubclass(BicicletaElectrica, Bicicleta))
print(BicicletaElectrica.__mro__)        # orden en que Python busca los métodos

### Polimorfismo y *duck typing*

Un mismo código funciona con objetos de clases distintas: cada uno responde a `tarifa()` a su manera. En Java eso exige que todos compartan una clase padre o una interfaz. En Python **no**: basta con que el objeto tenga el método. Es el ***duck typing***: "si camina como un pato y grazna como un pato, es un pato".

In [ ]:
class Patinete:                     # no hereda de Bicicleta
    def __init__(self, id):
        self.id = id

    def tarifa(self, minutos):
        return round(1 + minutos * 0.15, 2)


flota = [Bicicleta("B-11"), BicicletaElectrica("E-02"), Patinete("P-01")]
for vehiculo in flota:
    print(f"{type(vehiculo).__name__:<20}{vehiculo.tarifa(20):>6.2f} €")

print("Total:", sum(v.tarifa(20) for v in flota))

### Clases abstractas

A veces quieres **obligar** a que las subclases implementen un método, como con una interfaz o una clase `abstract` de Java. Para eso está el módulo `abc` (*abstract base classes*): una clase que hereda de `ABC` y tiene métodos con `@abstractmethod` **no se puede instanciar**, y sus subclases tampoco hasta que implementen todos esos métodos.

In [ ]:
from abc import ABC, abstractmethod


class Vehiculo(ABC):
    def __init__(self, id):
        self.id = id

    @abstractmethod
    def tarifa(self, minutos):
        ...

    def resumen(self, minutos):              # método normal, que usa el abstracto
        return f"{self.id}: {self.tarifa(minutos):.2f} € por {minutos} min"


class Moto(Vehiculo):
    def tarifa(self, minutos):
        return round(minutos * 0.26, 2)


class Coche(Vehiculo):                       # se le ha olvidado tarifa()
    pass


print(Moto("M-01").resumen(30))

for clase in (Vehiculo, Coche):
    try:
        clase("X")
    except TypeError as error:
        print("TypeError:", error)

### Ejercicio 3.1

La clase abstracta `Figura` ya está hecha. Crea tres subclases:
- `Rectangulo(ancho, alto)`.
- `Cuadrado(lado)`, que **hereda de `Rectangulo`** y llama a `super().__init__(lado, lado)`. No necesita implementar `area` ni `perimetro`: los hereda.
- `Circulo(radio)`. Usa `math.pi`.

In [ ]:
import math
from abc import ABC, abstractmethod


class Figura(ABC):
    @abstractmethod
    def area(self):
        ...

    @abstractmethod
    def perimetro(self):
        ...

    def describir(self):
        return f"{type(self).__name__}: área {self.area():.2f}, perímetro {self.perimetro():.2f}"


class Rectangulo(Figura):
    ...


class Cuadrado(Rectangulo):
    ...


class Circulo(Figura):
    ...


for figura in [Rectangulo(3, 4), Cuadrado(2), Circulo(1)]:
    print(figura.describir())

In [ ]:
# Comprobación
try:
    Figura()
    raise AssertionError("Figura debe seguir siendo abstracta")
except TypeError:
    pass
assert Rectangulo(3, 4).area() == 12 and Rectangulo(3, 4).perimetro() == 14
assert Cuadrado(5).area() == 25 and Cuadrado(5).perimetro() == 20
assert isinstance(Cuadrado(1), Rectangulo), "Cuadrado debe heredar de Rectangulo"
assert "area" not in Cuadrado.__dict__, "Cuadrado no necesita su propio area(): lo hereda"
assert round(Circulo(1).area(), 4) == 3.1416 and round(Circulo(2).perimetro(), 4) == 12.5664
assert Circulo(1).describir() == "Circulo: área 3.14, perímetro 6.28"
assert round(sum(f.area() for f in [Rectangulo(1, 2), Cuadrado(2), Circulo(1)]), 2) == 9.14
print("Correcto")

### Composición frente a herencia

La herencia expresa una relación **"es un"**: una bicicleta eléctrica *es una* bicicleta. La composición expresa **"tiene un"**: una estación *tiene* bicicletas. Una estación no es una lista de bicicletas, aunque las guarde en una.

Regla práctica: **prefiere la composición**. La herencia une mucho las clases (un cambio en el padre afecta a todas las hijas). Úsala solo cuando la relación "es un" sea clara.

In [ ]:
class Estacion:
    def __init__(self, nombre):
        self.nombre = nombre
        self.bicis = []                  # composición: la estación TIENE bicicletas

    def aparcar(self, bici):
        self.bicis.append(bici)

    def kilometros_totales(self):
        return sum(b.km for b in self.bicis)


alameda = Estacion("Alameda Principal")
alameda.aparcar(Bicicleta("B-20", km=150))
alameda.aparcar(BicicletaElectrica("E-05", km=90))
print(len(alameda.bicis), alameda.kilometros_totales())

### Ejercicio 3.2

Amplía la `Estacion` con una capacidad máxima:
- `__init__(self, nombre, capacidad)` guarda los dos valores y crea la lista **vacía** `self._bicis`.
- `aparcar(self, bici)` la añade al final. Si la estación está llena, lanza un `ValueError`.
- `retirar(self)` quita y **devuelve** la bicicleta que **más tiempo lleva** aparcada (la primera de la lista). Si no hay ninguna, lanza un `ValueError`.
- `__len__` devuelve cuántas bicicletas hay.
- La propiedad `libres` devuelve cuántos anclajes quedan libres.

In [ ]:
class Estacion:
    def __init__(self, nombre, capacidad):
        ...

    def aparcar(self, bici):
        ...

    def retirar(self):
        ...

    def __len__(self):
        ...

    @property
    def libres(self):
        ...


soho = Estacion("Soho", capacidad=2)
soho.aparcar(Bicicleta("B-30"))
print(len(soho), soho.libres)

In [ ]:
# Comprobación
e = Estacion("Prueba", capacidad=2)
assert len(e) == 0 and e.libres == 2
e.aparcar(Bicicleta("B-1"))
e.aparcar(BicicletaElectrica("E-1"))
assert len(e) == 2 and e.libres == 0
try:
    e.aparcar(Bicicleta("B-2"))
    raise AssertionError("Aparcar en una estación llena debe lanzar ValueError")
except ValueError:
    pass
primera = e.retirar()
assert isinstance(primera, Bicicleta) and primera.id == "B-1", "retirar() devuelve la que más tiempo lleva"
assert len(e) == 1 and e.libres == 1
e.retirar()
try:
    e.retirar()
    raise AssertionError("Retirar de una estación vacía debe lanzar ValueError")
except ValueError:
    pass
assert Estacion("Otra", 5)._bicis == [], "Cada estación debe tener su propia lista"
print("Correcto")

**Mixins (solo para que te suenen).** Python permite **herencia múltiple**: una clase puede heredar de varias. Su uso más habitual son los *mixins*, clases pequeñas que solo aportan un comportamiento concreto y que se "mezclan" con otras.

In [ ]:
import json


class JSONMixin:
    def a_json(self):
        return json.dumps(vars(self), ensure_ascii=False)


class Parada(JSONMixin):
    def __init__(self, nombre, linea):
        self.nombre = nombre
        self.linea = linea


print(Parada("Alameda Principal", 1).a_json())

## 4. Clases modernas

### *Dataclasses*

Muchas clases solo sirven para **guardar datos**. Escribir a mano su `__init__`, su `__repr__` y su `__eq__` es repetitivo (en Java lo resuelven los `record` o Lombok). El decorador `@dataclass` los genera a partir de los **atributos anotados con su tipo**.

In [ ]:
from dataclasses import dataclass, field


@dataclass
class Viaje:
    origen: str
    destino: str
    minutos: float
    etiquetas: list[str] = field(default_factory=list)   # una lista nueva para cada viaje


v1 = Viaje("Alameda", "Soho", 7)
v2 = Viaje("Alameda", "Soho", 7)
print(v1)                  # __repr__ generado
print(v1 == v2)            # __eq__ generado: compara los campos
v1.etiquetas.append("turista")
print(v1.etiquetas, v2.etiquetas)

Las opciones más útiles de `@dataclass`:

- `frozen=True`: el objeto es **inmutable** (no se pueden cambiar sus campos) y, a cambio, se puede usar como clave de un diccionario o dentro de un conjunto.
- `order=True`: genera `<`, `>`, `<=` y `>=`, que comparan los campos **en el orden en que están declarados**.
- `__post_init__`: un método que se ejecuta justo después del `__init__` generado. Se usa para **validar**.

In [ ]:
from dataclasses import FrozenInstanceError


@dataclass(frozen=True, order=True)
class Coordenada:
    latitud: float
    longitud: float

    def __post_init__(self):
        if not -90 <= self.latitud <= 90:
            raise ValueError(f"Latitud fuera de rango: {self.latitud}")


alameda = Coordenada(36.7178, -4.4254)
muelle = Coordenada(36.7176, -4.4130)

print(sorted([alameda, muelle]))                       # order=True
print({alameda, Coordenada(36.7178, -4.4254)})         # frozen=True: se puede meter en un conjunto

try:
    alameda.latitud = 0
except FrozenInstanceError as error:
    print("FrozenInstanceError:", error)

try:
    Coordenada(120, 0)
except ValueError as error:
    print("ValueError:", error)

### Enumeraciones: `Enum`

Un `Enum` define un conjunto **cerrado** de valores con nombre, igual que un `enum` de Java. Es mejor que usar cadenas sueltas como `"abonado"`: el editor autocompleta, y una errata da un error en lugar de pasar desapercibida.

`StrEnum` es un `Enum` cuyos miembros **también son cadenas**: muy cómodo para leer y escribir CSV o JSON.

In [ ]:
from enum import Enum, StrEnum


class Nivel(Enum):
    AMARILLO = 1
    NARANJA = 2
    ROJO = 3


class TipoUsuario(StrEnum):
    ABONADO = "abonado"
    OCASIONAL = "ocasional"


print(Nivel.ROJO, Nivel.ROJO.name, Nivel.ROJO.value)
print(list(TipoUsuario))
print(TipoUsuario("abonado") is TipoUsuario.ABONADO)   # desde el valor leído de un fichero
print(TipoUsuario.ABONADO == "abonado", f"{TipoUsuario.OCASIONAL}")

try:
    TipoUsuario("turista")
except ValueError as error:
    print("ValueError:", error)

### Ejercicio 4.1

Crea la *dataclass* `Alquiler` con `order=True` y estos campos, en este orden: `minutos` (`float`), `origen` (`str`), `destino` (`str`) y `tipo` (`TipoUsuario`, por defecto `TipoUsuario.OCASIONAL`).
- En `__post_init__`, lanza un `ValueError` si `minutos` no es mayor que 0.
- Añade la propiedad `coste`:
  - Ocasional: 0,05 € por minuto.
  - Abonado: los 30 primeros minutos son gratis y el resto se paga a 0,05 € por minuto.
  - Redondea a 2 decimales.

In [ ]:
@dataclass(order=True)
class Alquiler:
    ...


a = Alquiler(12, "Soho", "Muelle Uno")
print(a, a.coste)

In [ ]:
# Comprobación
import dataclasses

assert dataclasses.is_dataclass(Alquiler), "Usa @dataclass(order=True)"
assert [f.name for f in dataclasses.fields(Alquiler)] == ["minutos", "origen", "destino", "tipo"]
a = Alquiler(12, "Soho", "Muelle Uno")
assert a.tipo is TipoUsuario.OCASIONAL and a.coste == 0.6
assert repr(a).startswith("Alquiler(minutos=12, origen='Soho'")
assert Alquiler(20, "A", "B", TipoUsuario.ABONADO).coste == 0
assert Alquiler(40, "A", "B", TipoUsuario.ABONADO).coste == 0.5
assert sorted([Alquiler(30, "A", "B"), Alquiler(5, "C", "D")])[0].minutos == 5, "Falta order=True"
for mal in (0, -3):
    try:
        Alquiler(mal, "A", "B")
        raise AssertionError("Una duración <= 0 debe lanzar ValueError")
    except ValueError:
        pass
print("Correcto")

### Ejercicio 4.2

Las `Alerta` meteorológicas llegan **repetidas** desde varias fuentes.
- Haz que `Alerta` sea una *dataclass* **inmutable**, con los campos `zona` (`str`) y `nivel` (`Nivel`).
- Guarda en `unicas` las alertas sin repetir. Pista: un conjunto.
- Guarda en `peor` la alerta de nivel más alto. Pista: `max` con `key` y el `.value` del nivel.

In [ ]:
class Alerta:
    ...


recibidas = [
    Alerta("Costa del Sol", Nivel.AMARILLO),
    Alerta("Axarquía", Nivel.NARANJA),
    Alerta("Costa del Sol", Nivel.AMARILLO),
    Alerta("Serranía de Ronda", Nivel.ROJO),
    Alerta("Axarquía", Nivel.NARANJA),
]
unicas = ...
peor = ...
print(len(unicas), peor)

In [ ]:
# Comprobación
assert len(unicas) == 3
assert peor == Alerta("Serranía de Ronda", Nivel.ROJO)
try:
    peor.nivel = Nivel.AMARILLO
    raise AssertionError("Alerta debe ser inmutable: usa frozen=True")
except dataclasses.FrozenInstanceError:
    pass
print("Correcto")

### Anotaciones de tipo (*type hints*)

Ya las has usado en las *dataclasses*. Se pueden poner en cualquier variable, parámetro o valor de retorno:

| Anotación | Significa |
|-----------|-----------|
| `int`, `float`, `str`, `bool` | Tipos básicos |
| `list[int]`, `dict[str, float]`, `tuple[str, ...]` | Colecciones con el tipo de sus elementos |
| `str \| None` | Una cadena o `None` (en Java, un valor que puede ser `null`) |
| `-> float` | Lo que devuelve la función |

**Python no las comprueba al ejecutar.** Son documentación que entienden el editor (autocompletado, avisos) y herramientas como **mypy**.

In [ ]:
def precio(minutos: float, tarifa: float = 0.05) -> float:
    return round(minutos * tarifa, 2)


def buscar(nombres: list[str], texto: str) -> str | None:
    for nombre in nombres:
        if texto.lower() in nombre.lower():
            return nombre
    return None


print(precio(20), buscar(["Alameda", "Soho"], "so"), buscar(["Alameda"], "palo"))

try:
    print(precio("20", 3))   # Python no se queja al llamarla: "20" * 3 es "202020"...
except TypeError as error:   # ...y el error salta después, dentro de round()
    print("TypeError:", error)

La llamada `precio("20", 3)` viola las anotaciones, pero Python solo lo descubre **al ejecutarla**, y además el error aparece lejos de su causa. Un comprobador de tipos como **mypy** encuentra estos errores **sin ejecutar el código**. Pruébalo (necesita conexión para instalarlo):

In [ ]:
%%writefile tipos_demo.py
def etiqueta(nombre: str | None) -> str:
    return nombre.upper()


print(etiqueta("bici"))

In [ ]:
!"{sys.executable}" tipos_demo.py
!"{sys.executable}" -m pip install --quiet --disable-pip-version-check mypy
!"{sys.executable}" -m mypy tipos_demo.py

El programa funciona con `"bici"`, pero mypy avisa de que `nombre` **puede ser `None`** y entonces `.upper()` fallaría. En un proyecto grande, este tipo de avisos evita muchos errores.

### `Protocol`: interfaces con *duck typing*

Un `Protocol` describe **qué métodos** debe tener un objeto, sin obligarle a heredar de nada. Es la versión con tipos del *duck typing*: mypy comprueba que los objetos encajan, pero en tiempo de ejecución no cambia nada.

In [ ]:
from typing import Protocol


class Tarificable(Protocol):
    def tarifa(self, minutos: float) -> float: ...


def total(vehiculos: list[Tarificable], minutos: float) -> float:
    return round(sum(v.tarifa(minutos) for v in vehiculos), 2)


# Bicicleta, BicicletaElectrica y Patinete no heredan de Tarificable, pero encajan
print(total([Bicicleta("B-1"), BicicletaElectrica("E-1"), Patinete("P-1")], 10))

## 5. Práctica final · Catálogo de datasets

La práctica no se hace en el notebook, sino en un **archivo de Python**, que es como se trabaja con clases en un proyecto real. Está en la carpeta `ejercicios/catalogo/` del material:

| Archivo | Contenido |
|---------|-----------|
| `catalogo.py` | El esqueleto que tienes que completar, con instrucciones en los comentarios |
| `datos/datasets.json` | 30 fichas de un catálogo de datos abiertos. **Son datos simulados** y 4 de ellas tienen errores a propósito |
| `comprobar.py` | Comprueba tu solución. Ejecútalo con `python3 comprobar.py` (en Windows, `python comprobar.py`) |

Vas a modelar el catálogo con todo lo que has visto hoy:

- **A.** El `Enum` `Formato` y la *dataclass* inmutable `Dataset`, con validación en `__post_init__` y una propiedad.
- **B.** Los métodos `Dataset.desde_dict()` (método de clase) y `a_dict()`, para convertir desde y hacia JSON.
- **C.** La clase `Catalogo`, que **contiene** datasets (composición), con una excepción propia y los métodos especiales `__len__`, `__iter__`, `__contains__`, `__getitem__` y `__repr__`.
- **D.** Consultas: buscar, filtrar por formato, los más recientes y un resumen.
- **E.** Cargar el catálogo desde el JSON, descartando las fichas con errores, y guardarlo.

Ve parte a parte y ejecuta `comprobar.py` al terminar cada una.

## Para seguir practicando

- Añade a `Catalogo` un método `organismos()` que devuelva el **conjunto** de organismos que publican datos, y otro `por_organismo()` que devuelva un diccionario `{organismo: [datasets]}`.
- Haz que `Dataset` tenga `order=True`. ¿Por qué campo ordena? ¿Te sirve? Prueba a ordenar con `sorted(catalogo, key=...)` en su lugar.
- Pasa **mypy** sobre tu `catalogo.py` y corrige los avisos que salgan.
- Crea una subclase `CatalogoMunicipal(Catalogo)` que solo acepte datasets del Ayuntamiento y lance un `ValueError` con los demás. Usa `super().agregar(...)`.